# Kapak Real Process Indices

Pipeline end-to-end para:
- entrenar NB, RF y LR,
- predecir probabilidades sobre embeddings reales de Kapak,
- calcular ?ndices por `contract_id` real (sin fusi?n de modelos).


In [1]:
from dataclasses import dataclass
from pathlib import Path
from typing import List

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB


In [2]:
@dataclass
class Paths:
    train_nb: Path
    train_rf: Path
    train_lr: Path
    target_embeddings: Path
    target_raw: Path
    output_dir: Path


def default_paths(base_dir: Path) -> Paths:
    return Paths(
        train_nb=base_dir
        / "Codes"
        / "Archivos_Naive_Bayes"
        / "train_embeddings_balanced_AUG_GPT4o_mini_total_augmented_text-embedding-3-large.csv",
        train_rf=base_dir
        / "Codes"
        / "Archivos_Random_Forest"
        / "train_embeddings_balanced_total_sentence_prompt_GPT4o_mini_text-embedding-3-large.csv",
        train_lr=base_dir
        / "Codes"
        / "Archivos_Logistic_Regression"
        / "train_embeddings_balanced_total_sinonimos_text-embedding-3-large.csv",
        target_embeddings=base_dir
        / "Codes"
        / "output"
        / "kapak_sie_embeddings_matrix_text_embedding_3_large.csv",
        target_raw=base_dir / "Codes" / "output" / "kapak_sie_questions_raw.csv",
        output_dir=base_dir / "Codes" / "output",
    )


def _validate_files(paths: Paths) -> None:
    for p in [
        paths.train_nb,
        paths.train_rf,
        paths.train_lr,
        paths.target_embeddings,
        paths.target_raw,
    ]:
        if not p.exists():
            raise FileNotFoundError(f"No se encontro archivo requerido: {p}")


In [3]:
def _feature_cols(df: pd.DataFrame) -> List[str]:
    return [c for c in df.columns if c != "label"]


def _load_target_embeddings(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    if "label" in df.columns:
        df = df.drop(columns=["label"])
    return df


def _load_target_contract_ids(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    if "contract_id" not in df.columns:
        raise ValueError("El archivo raw no contiene la columna 'contract_id'.")
    return df[["contract_id"]].copy()


def _prepare_model_inputs(
    train_path: Path, target_embeddings: pd.DataFrame
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    train = pd.read_csv(train_path)
    if "label" not in train.columns:
        raise ValueError(f"El train no tiene columna label: {train_path}")

    features = _feature_cols(train)
    missing = [c for c in features if c not in target_embeddings.columns]
    if missing:
        raise ValueError(
            f"Faltan columnas de embeddings en target para {train_path.name}. "
            f"Ejemplos faltantes: {missing[:10]}"
        )

    x_train = train[features].to_numpy()
    y_train = train["label"].to_numpy()
    x_target = target_embeddings[features].to_numpy()
    return x_train, y_train, x_target


In [4]:
def _noisy_or(arr: np.ndarray) -> float:
    """Noisy-OR probabilistic accumulation: 1 − ∏(1 − pᵢ).
    
    Interpreta cada probabilidad como una señal independiente de riesgo.
    El índice crece con cada señal adicional y nunca decrece.
    """
    if arr.size == 0:
        return float("nan")
    return float(1.0 - np.prod(1.0 - np.clip(arr, 0.0, 1.0)))

In [5]:
# Configuracion de rutas
base_dir = Path.cwd().resolve().parent if Path.cwd().name.lower() == "codes" else Path.cwd().resolve()
paths = default_paths(base_dir)
_validate_files(paths)

print(paths)


Paths(train_nb=WindowsPath('D:/Noveno Semestre/Tesis Informal/Codes/Archivos_Naive_Bayes/train_embeddings_balanced_AUG_GPT4o_mini_total_augmented_text-embedding-3-large.csv'), train_rf=WindowsPath('D:/Noveno Semestre/Tesis Informal/Codes/Archivos_Random_Forest/train_embeddings_balanced_total_sentence_prompt_GPT4o_mini_text-embedding-3-large.csv'), train_lr=WindowsPath('D:/Noveno Semestre/Tesis Informal/Codes/Archivos_Logistic_Regression/train_embeddings_balanced_total_sinonimos_text-embedding-3-large.csv'), target_embeddings=WindowsPath('D:/Noveno Semestre/Tesis Informal/Codes/output/kapak_sie_embeddings_matrix_text_embedding_3_large.csv'), target_raw=WindowsPath('D:/Noveno Semestre/Tesis Informal/Codes/output/kapak_sie_questions_raw.csv'), output_dir=WindowsPath('D:/Noveno Semestre/Tesis Informal/Codes/output'))


In [6]:
# Carga de datos target
paths.output_dir.mkdir(parents=True, exist_ok=True)

target_embeddings = _load_target_embeddings(paths.target_embeddings)
target_raw = _load_target_contract_ids(paths.target_raw)

if len(target_embeddings) != len(target_raw):
    raise ValueError(
        "No coincide el numero de filas entre embeddings y raw. "
        f"embeddings={len(target_embeddings)}, raw={len(target_raw)}"
    )

print("Filas target:", len(target_embeddings))


Filas target: 1659


In [7]:
# Modelo Naive Bayes
x_train_nb, y_train_nb, x_target_nb = _prepare_model_inputs(paths.train_nb, target_embeddings)
model_nb = GaussianNB(var_smoothing=0.35111917342151305)
model_nb.fit(x_train_nb, y_train_nb)
prob_nb = model_nb.predict_proba(x_target_nb)[:, 1]


In [8]:
# Modelo Random Forest
x_train_rf, y_train_rf, x_target_rf = _prepare_model_inputs(paths.train_rf, target_embeddings)
model_rf = RandomForestClassifier(
    max_depth=None,
    min_samples_leaf=1,
    n_estimators=200,
    random_state=42,
    n_jobs=1,
)
model_rf.fit(x_train_rf, y_train_rf)
prob_rf = model_rf.predict_proba(x_target_rf)[:, 1]


In [9]:
# Modelo Logistic Regression
x_train_lr, y_train_lr, x_target_lr = _prepare_model_inputs(paths.train_lr, target_embeddings)
model_lr = LogisticRegression(
    C=100,
    penalty="l2",
    solver="saga",
    max_iter=1000,
    random_state=42,
)
model_lr.fit(x_train_lr, y_train_lr)
prob_lr = model_lr.predict_proba(x_target_lr)[:, 1]


In [10]:
# Probabilidades por pregunta
q_probs = target_raw.copy()
q_probs.insert(0, "row_id", np.arange(len(q_probs), dtype=int))
q_probs["prob_nb"] = prob_nb
q_probs["prob_rf"] = prob_rf
q_probs["prob_lr"] = prob_lr

q_probs.head(10)


,row_id,contract_id,prob_nb,prob_rf,prob_lr
0,0,1666287,8.460365e-04,0.125,0.002336
1,1,1666287,7.953427e-12,0.070,0.000003
2,2,1666287,1.000000e+00,0.265,0.115301
3,3,1666287,5.081304e-14,0.060,0.000026
4,4,1676214,1.000000e+00,0.275,0.026318
5,5,1676214,3.742756e-01,0.190,0.000533
6,6,1676214,8.552782e-01,0.175,0.000685
7,7,1676214,1.119952e-04,0.095,0.000097
8,8,1676214,1.000000e+00,0.220,0.009230
9,9,1676214,1.000000e+00,0.440,0.514710


## Cálculo de Índices de Riesgo por Proceso

Se implementan **4 métodos** organizados en 2 tracks. Todos son completamente probabilísticos (no requieren umbralización binaria).

| Track | Método | Descripción |
|-------|--------|-------------|
| **A** | M1 — Media por modelo | `mean(p_m)` por modelo → promedio de 3 modelos |
| **A** | M2 — Noisy-OR por modelo | `noisy_or(p_m)` por modelo → promedio de 3 modelos |
| **B** | M3 — Soft Vote → Media | `r_q = mean(p_nb, p_rf, p_lr)` por pregunta → `mean(r_q)` |
| **B** | M4 — Soft Vote → Noisy-OR | `r_q = mean(p_nb, p_rf, p_lr)` por pregunta → `noisy_or(r_q)` |

In [13]:
# ══════════════════════════════════════════════════════════════════════════════
# TRACK A — Modelos separados → combinar a nivel de proceso
# ══════════════════════════════════════════════════════════════════════════════
# Cada modelo opera de forma independiente sobre sus propias probabilidades.
# El índice final del proceso es el promedio de los 3 scores de modelo.
#
# M1: Media aritmética  →  IC_M1 = (mean_nb + mean_rf + mean_lr) / 3
# M2: Noisy-OR          →  IC_M2 = (nor_nb  + nor_rf  + nor_lr ) / 3

def _track_a(df_q: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for contract_id, g in df_q.groupby("contract_id", sort=True):
        p_nb = g["prob_nb"].to_numpy(dtype=float)
        p_rf = g["prob_rf"].to_numpy(dtype=float)
        p_lr = g["prob_lr"].to_numpy(dtype=float)

        # M1 — media por modelo
        m1_nb, m1_rf, m1_lr = p_nb.mean(), p_rf.mean(), p_lr.mean()

        # M2 — Noisy-OR por modelo
        m2_nb = _noisy_or(p_nb)
        m2_rf = _noisy_or(p_rf)
        m2_lr = _noisy_or(p_lr)

        rows.append({
            "contract_id": contract_id,
            "n_preguntas": len(g),
            # intermedios por modelo (útiles para diagnóstico)
            "M1_nb": m1_nb, "M1_rf": m1_rf, "M1_lr": m1_lr,
            "M2_nb": m2_nb, "M2_rf": m2_rf, "M2_lr": m2_lr,
            # índices finales Track A
            "IC_M1": (m1_nb + m1_rf + m1_lr) / 3.0,
            "IC_M2": (m2_nb + m2_rf + m2_lr) / 3.0,
        })
    return pd.DataFrame(rows)

track_a = _track_a(q_probs)

print("TRACK A — Índices por proceso (M1: Media, M2: Noisy-OR)")
display(track_a[["contract_id", "n_preguntas", "M1_nb", "M1_rf", "M1_lr", "IC_M1",
                                                "M2_nb", "M2_rf", "M2_lr", "IC_M2"]].head(20))

# ══════════════════════════════════════════════════════════════════════════════
# TRACK B — Soft Voting a nivel de pregunta → agregar a nivel de proceso
# ══════════════════════════════════════════════════════════════════════════════
# Paso 1: combinar los 3 modelos por pregunta (soft vote = promedio de probs).
#   r_q = (p_nb + p_rf + p_lr) / 3
# Paso 2: agregar r_q al nivel de proceso.
#
# M3: Soft Vote → Media    →  IC_M3 = mean(r_q)
# M4: Soft Vote → Noisy-OR →  IC_M4 = 1 − ∏(1 − r_q)

def _track_b(df_q: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for contract_id, g in df_q.groupby("contract_id", sort=True):
        p_nb = g["prob_nb"].to_numpy(dtype=float)
        p_rf = g["prob_rf"].to_numpy(dtype=float)
        p_lr = g["prob_lr"].to_numpy(dtype=float)

        # Soft vote por pregunta
        r_q = (p_nb + p_rf + p_lr) / 3.0

        rows.append({
            "contract_id": contract_id,
            "n_preguntas": len(g),
            "IC_M3": r_q.mean(),
            "IC_M4": _noisy_or(r_q),
        })
    return pd.DataFrame(rows)

track_b = _track_b(q_probs)

print("\nTRACK B — Índices por proceso (M3: Vote→Media, M4: Vote→Noisy-OR)")
display(track_b.head(20))

# ══════════════════════════════════════════════════════════════════════════════
# TABLA CONSOLIDADA — 4 índices por proceso
# ══════════════════════════════════════════════════════════════════════════════

indices_finales = (
    track_a[["contract_id", "n_preguntas", "IC_M1", "IC_M2"]]
    .merge(track_b[["contract_id", "IC_M3", "IC_M4"]], on="contract_id", how="outer")
    .sort_values("contract_id")
    .reset_index(drop=True)
)

print("\nTABLA CONSOLIDADA — 4 métodos por proceso")
display(indices_finales.head(20))

out_path = paths.output_dir / "kapak_sie_process_indices_4methods.csv"
indices_finales.to_csv(out_path, index=False, encoding="utf-8")
print(f"\n[OK] Guardado en: {out_path}")

TRACK A — Índices por proceso (M1: Media, M2: Noisy-OR)


,contract_id,n_preguntas,M1_nb,M1_rf,M1_lr,IC_M1,M2_nb,M2_rf,M2_lr,IC_M2
0,1666287,4,2.502115e-01,0.130000,0.029417,0.136543,1.000000e+00,0.437780,0.117395,0.518392
1,1676214,42,3.805535e-01,0.154167,0.019394,0.184705,1.000000e+00,0.999277,0.641959,0.880412
2,1676219,2,1.319054e-13,0.040000,0.000034,0.013345,2.637890e-13,0.078400,0.000069,0.026156
3,1676223,18,2.722581e-03,0.113611,0.001074,0.039136,4.830724e-02,0.887518,0.019177,0.318334
4,1676242,9,2.222199e-01,0.114444,0.020453,0.119039,1.000000e+00,0.686381,0.183917,0.623433
5,1676251,3,1.000000e+00,0.273333,0.304861,0.526065,1.000000e+00,0.616680,0.666305,0.760995
6,1676254,4,4.779614e-05,0.088750,0.000169,0.029656,1.911842e-04,0.311383,0.000678,0.104084
7,1676255,10,5.915660e-01,0.192000,0.046296,0.276621,1.000000e+00,0.890288,0.420600,0.770296
8,1676563,3,5.727571e-11,0.091667,0.000288,0.030652,1.718271e-10,0.251135,0.000864,0.084000
9,1676633,8,2.500009e-01,0.173750,0.087943,0.170565,1.000000e+00,0.826800,0.689686,0.838829



TRACK B — Índices por proceso (M3: Vote→Media, M4: Vote→Noisy-OR)


,contract_id,n_preguntas,IC_M3,IC_M4
0,1666287,4,0.136543,0.505329
1,1676214,42,0.184705,0.999941
2,1676219,2,0.013345,0.026511
3,1676223,18,0.039136,0.513364
4,1676242,9,0.119039,0.751536
5,1676251,3,0.526065,0.893878
6,1676254,4,0.029656,0.113564
7,1676255,10,0.276621,0.973653
8,1676563,3,0.030652,0.089234
9,1676633,8,0.170565,0.883649



TABLA CONSOLIDADA — 4 métodos por proceso


,contract_id,n_preguntas,IC_M1,IC_M2,IC_M3,IC_M4
0,1666287,4,0.136543,0.518392,0.136543,0.505329
1,1676214,42,0.184705,0.880412,0.184705,0.999941
2,1676219,2,0.013345,0.026156,0.013345,0.026511
3,1676223,18,0.039136,0.318334,0.039136,0.513364
4,1676242,9,0.119039,0.623433,0.119039,0.751536
5,1676251,3,0.526065,0.760995,0.526065,0.893878
6,1676254,4,0.029656,0.104084,0.029656,0.113564
7,1676255,10,0.276621,0.770296,0.276621,0.973653
8,1676563,3,0.030652,0.084000,0.030652,0.089234
9,1676633,8,0.170565,0.838829,0.170565,0.883649



[OK] Guardado en: D:\Noveno Semestre\Tesis Informal\Codes\output\kapak_sie_process_indices_4methods.csv


In [16]:
def _track_a(df_q: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for contract_id, g in df_q.groupby("contract_id", sort=True):
        p_nb = g["prob_nb"].to_numpy(dtype=float)
        p_rf = g["prob_rf"].to_numpy(dtype=float)
        p_lr = g["prob_lr"].to_numpy(dtype=float)

        rows.append({
            "contract_id": contract_id,
            "n_preguntas": len(g),
            # M1 — media por modelo (3 índices independientes)
            "IC_M1_nb": p_nb.mean(),
            "IC_M1_rf": p_rf.mean(),
            "IC_M1_lr": p_lr.mean(),
            # M2 — Noisy-OR por modelo (3 índices independientes)
            "IC_M2_nb": _noisy_or(p_nb),
            "IC_M2_rf": _noisy_or(p_rf),
            "IC_M2_lr": _noisy_or(p_lr),
        })
    return pd.DataFrame(rows)

track_a = _track_a(q_probs)

print("TRACK A — Índices por proceso, por modelo (sin fusión)")
display(track_a.head(20))

# ══════════════════════════════════════════════════════════════════════════════
# TRACK B — Soft Voting a nivel de pregunta → agregar a nivel de proceso
# ══════════════════════════════════════════════════════════════════════════════
# Paso 1: combinar los 3 modelos por pregunta (soft vote = promedio de probs).
#   r_q = (p_nb + p_rf + p_lr) / 3
# Paso 2: agregar r_q al nivel de proceso.
#
# M3: Soft Vote → Media    →  IC_M3 = mean(r_q)
# M4: Soft Vote → Noisy-OR →  IC_M4 = 1 − ∏(1 − r_q)

def _track_b(df_q: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for contract_id, g in df_q.groupby("contract_id", sort=True):
        p_nb = g["prob_nb"].to_numpy(dtype=float)
        p_rf = g["prob_rf"].to_numpy(dtype=float)
        p_lr = g["prob_lr"].to_numpy(dtype=float)

        # Soft vote por pregunta
        r_q = (p_nb + p_rf + p_lr) / 3.0

        rows.append({
            "contract_id": contract_id,
            "n_preguntas": len(g),
            "IC_M3": r_q.mean(),
            "IC_M4": _noisy_or(r_q),
        })
    return pd.DataFrame(rows)

track_b = _track_b(q_probs)

print("\nTRACK B — Índices por proceso (M3: Vote→Media, M4: Vote→Noisy-OR)")
display(track_b.head(20))

# ══════════════════════════════════════════════════════════════════════════════
# TABLA CONSOLIDADA — todos los índices por proceso
# ══════════════════════════════════════════════════════════════════════════════

indices_finales = (
    track_a
    .merge(track_b[["contract_id", "IC_M3", "IC_M4"]], on="contract_id", how="outer")
    .sort_values("contract_id")
    .reset_index(drop=True)
)

print("\nTABLA CONSOLIDADA")
display(indices_finales.head(20))

out_path = paths.output_dir / "kapak_sie_process_indices_4methods.csv"
indices_finales.to_csv(out_path, index=False, encoding="utf-8")
print(f"\n[OK] Guardado en: {out_path}")

TRACK A — Índices por proceso, por modelo (sin fusión)


,contract_id,n_preguntas,IC_M1_nb,IC_M1_rf,IC_M1_lr,IC_M2_nb,IC_M2_rf,IC_M2_lr
0,1666287,4,2.502115e-01,0.130000,0.029417,1.000000e+00,0.437780,0.117395
1,1676214,42,3.805535e-01,0.154167,0.019394,1.000000e+00,0.999277,0.641959
2,1676219,2,1.319054e-13,0.040000,0.000034,2.637890e-13,0.078400,0.000069
3,1676223,18,2.722581e-03,0.113611,0.001074,4.830724e-02,0.887518,0.019177
4,1676242,9,2.222199e-01,0.114444,0.020453,1.000000e+00,0.686381,0.183917
5,1676251,3,1.000000e+00,0.273333,0.304861,1.000000e+00,0.616680,0.666305
6,1676254,4,4.779614e-05,0.088750,0.000169,1.911842e-04,0.311383,0.000678
7,1676255,10,5.915660e-01,0.192000,0.046296,1.000000e+00,0.890288,0.420600
8,1676563,3,5.727571e-11,0.091667,0.000288,1.718271e-10,0.251135,0.000864
9,1676633,8,2.500009e-01,0.173750,0.087943,1.000000e+00,0.826800,0.689686



TRACK B — Índices por proceso (M3: Vote→Media, M4: Vote→Noisy-OR)


,contract_id,n_preguntas,IC_M3,IC_M4
0,1666287,4,0.136543,0.505329
1,1676214,42,0.184705,0.999941
2,1676219,2,0.013345,0.026511
3,1676223,18,0.039136,0.513364
4,1676242,9,0.119039,0.751536
5,1676251,3,0.526065,0.893878
6,1676254,4,0.029656,0.113564
7,1676255,10,0.276621,0.973653
8,1676563,3,0.030652,0.089234
9,1676633,8,0.170565,0.883649



TABLA CONSOLIDADA


,contract_id,n_preguntas,IC_M1_nb,IC_M1_rf,IC_M1_lr,IC_M2_nb,IC_M2_rf,IC_M2_lr,IC_M3,IC_M4
0,1666287,4,2.502115e-01,0.130000,0.029417,1.000000e+00,0.437780,0.117395,0.136543,0.505329
1,1676214,42,3.805535e-01,0.154167,0.019394,1.000000e+00,0.999277,0.641959,0.184705,0.999941
2,1676219,2,1.319054e-13,0.040000,0.000034,2.637890e-13,0.078400,0.000069,0.013345,0.026511
3,1676223,18,2.722581e-03,0.113611,0.001074,4.830724e-02,0.887518,0.019177,0.039136,0.513364
4,1676242,9,2.222199e-01,0.114444,0.020453,1.000000e+00,0.686381,0.183917,0.119039,0.751536
5,1676251,3,1.000000e+00,0.273333,0.304861,1.000000e+00,0.616680,0.666305,0.526065,0.893878
6,1676254,4,4.779614e-05,0.088750,0.000169,1.911842e-04,0.311383,0.000678,0.029656,0.113564
7,1676255,10,5.915660e-01,0.192000,0.046296,1.000000e+00,0.890288,0.420600,0.276621,0.973653
8,1676563,3,5.727571e-11,0.091667,0.000288,1.718271e-10,0.251135,0.000864,0.030652,0.089234
9,1676633,8,2.500009e-01,0.173750,0.087943,1.000000e+00,0.826800,0.689686,0.170565,0.883649



[OK] Guardado en: D:\Noveno Semestre\Tesis Informal\Codes\output\kapak_sie_process_indices_4methods.csv


In [17]:
# ══════════════════════════════════════════════════════════════════════════════
# TRACK A — Modelos separados (sin fusión)
# ══════════════════════════════════════════════════════════════════════════════
# Cada modelo produce su propio índice de forma independiente.
# No se combina entre modelos: el resultado son 3 columnas por método.
#
# M1: Media aritmética  →  IC_M1_nb, IC_M1_rf, IC_M1_lr
# M2: Noisy-OR          →  IC_M2_nb, IC_M2_rf, IC_M2_lr

def _track_a(df_q: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for contract_id, g in df_q.groupby("contract_id", sort=True):
        p_nb = g["prob_nb"].to_numpy(dtype=float)
        p_rf = g["prob_rf"].to_numpy(dtype=float)
        p_lr = g["prob_lr"].to_numpy(dtype=float)

        rows.append({
            "contract_id": contract_id,
            "n_preguntas": len(g),
            # M1 — media por modelo (3 índices independientes)
            "IC_M1_nb": p_nb.mean(),
            "IC_M1_rf": p_rf.mean(),
            "IC_M1_lr": p_lr.mean(),
            # M2 — Noisy-OR por modelo (3 índices independientes)
            "IC_M2_nb": _noisy_or(p_nb),
            "IC_M2_rf": _noisy_or(p_rf),
            "IC_M2_lr": _noisy_or(p_lr),
        })
    return pd.DataFrame(rows)

track_a = _track_a(q_probs)

print("TRACK A — Índices por proceso, por modelo (sin fusión)")
display(track_a.head(20))

# ══════════════════════════════════════════════════════════════════════════════
# TRACK B — Soft Voting ponderado por Brier → agregar a nivel de proceso
# ══════════════════════════════════════════════════════════════════════════════
# Los pesos se derivan del Brier score de cada modelo en el test set:
# a menor Brier score (mejor calibración), mayor peso.
# Fuente: modelos_probabilidades.ipynb
#
# Pesos normalizados (suman 1):
#   w_lr = 0.5478   w_rf = 0.3998   w_nb = 0.0524
#
# Paso 1: soft vote ponderado por pregunta
#   r_q = w_lr·p_lr + w_rf·p_rf + w_nb·p_nb
# Paso 2: agregar r_q al nivel de proceso
#
# M3: Soft Vote → Media    →  IC_M3 = mean(r_q)
# M4: Soft Vote → Noisy-OR →  IC_M4 = 1 − ∏(1 − r_q)

W_LR = 0.5478240336268073
W_RF = 0.3998231918002441
W_NB = 0.05235277457294863

def _track_b(df_q: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for contract_id, g in df_q.groupby("contract_id", sort=True):
        p_nb = g["prob_nb"].to_numpy(dtype=float)
        p_rf = g["prob_rf"].to_numpy(dtype=float)
        p_lr = g["prob_lr"].to_numpy(dtype=float)

        # Soft vote ponderado por Brier (por pregunta)
        r_q = W_LR * p_lr + W_RF * p_rf + W_NB * p_nb

        rows.append({
            "contract_id": contract_id,
            "n_preguntas": len(g),
            "IC_M3": r_q.mean(),
            "IC_M4": _noisy_or(r_q),
        })
    return pd.DataFrame(rows)

track_b = _track_b(q_probs)

print(f"\nTRACK B — Soft Vote ponderado por Brier (w_lr={W_LR:.4f}, w_rf={W_RF:.4f}, w_nb={W_NB:.4f})")
print("Índices por proceso (M3: Vote→Media, M4: Vote→Noisy-OR)")
display(track_b.head(20))

# ══════════════════════════════════════════════════════════════════════════════
# TABLA CONSOLIDADA — todos los índices por proceso
# ══════════════════════════════════════════════════════════════════════════════

indices_finales = (
    track_a
    .merge(track_b[["contract_id", "IC_M3", "IC_M4"]], on="contract_id", how="outer")
    .sort_values("contract_id")
    .reset_index(drop=True)
)

print("\nTABLA CONSOLIDADA")
display(indices_finales.head(20))

out_path = paths.output_dir / "kapak_sie_process_indices_4methods.csv"
indices_finales.to_csv(out_path, index=False, encoding="utf-8")
print(f"\n[OK] Guardado en: {out_path}")

TRACK A — Índices por proceso, por modelo (sin fusión)


,contract_id,n_preguntas,IC_M1_nb,IC_M1_rf,IC_M1_lr,IC_M2_nb,IC_M2_rf,IC_M2_lr
0,1666287,4,2.502115e-01,0.130000,0.029417,1.000000e+00,0.437780,0.117395
1,1676214,42,3.805535e-01,0.154167,0.019394,1.000000e+00,0.999277,0.641959
2,1676219,2,1.319054e-13,0.040000,0.000034,2.637890e-13,0.078400,0.000069
3,1676223,18,2.722581e-03,0.113611,0.001074,4.830724e-02,0.887518,0.019177
4,1676242,9,2.222199e-01,0.114444,0.020453,1.000000e+00,0.686381,0.183917
5,1676251,3,1.000000e+00,0.273333,0.304861,1.000000e+00,0.616680,0.666305
6,1676254,4,4.779614e-05,0.088750,0.000169,1.911842e-04,0.311383,0.000678
7,1676255,10,5.915660e-01,0.192000,0.046296,1.000000e+00,0.890288,0.420600
8,1676563,3,5.727571e-11,0.091667,0.000288,1.718271e-10,0.251135,0.000864
9,1676633,8,2.500009e-01,0.173750,0.087943,1.000000e+00,0.826800,0.689686



TRACK B — Soft Vote ponderado por Brier (w_lr=0.5478, w_rf=0.3998, w_nb=0.0524)
Índices por proceso (M3: Vote→Media, M4: Vote→Noisy-OR)


,contract_id,n_preguntas,IC_M3,IC_M4
0,1666287,4,0.081192,0.299316
1,1676214,42,0.092187,0.986075
2,1676219,2,0.016012,0.031767
3,1676223,18,0.046155,0.573688
4,1676242,9,0.068596,0.496350
5,1676251,3,0.328648,0.698488
6,1676254,4,0.035580,0.135063
7,1676255,10,0.133098,0.778757
8,1676563,3,0.036808,0.106509
9,1676633,8,0.130735,0.773112



TABLA CONSOLIDADA


,contract_id,n_preguntas,IC_M1_nb,IC_M1_rf,IC_M1_lr,IC_M2_nb,IC_M2_rf,IC_M2_lr,IC_M3,IC_M4
0,1666287,4,2.502115e-01,0.130000,0.029417,1.000000e+00,0.437780,0.117395,0.081192,0.299316
1,1676214,42,3.805535e-01,0.154167,0.019394,1.000000e+00,0.999277,0.641959,0.092187,0.986075
2,1676219,2,1.319054e-13,0.040000,0.000034,2.637890e-13,0.078400,0.000069,0.016012,0.031767
3,1676223,18,2.722581e-03,0.113611,0.001074,4.830724e-02,0.887518,0.019177,0.046155,0.573688
4,1676242,9,2.222199e-01,0.114444,0.020453,1.000000e+00,0.686381,0.183917,0.068596,0.496350
5,1676251,3,1.000000e+00,0.273333,0.304861,1.000000e+00,0.616680,0.666305,0.328648,0.698488
6,1676254,4,4.779614e-05,0.088750,0.000169,1.911842e-04,0.311383,0.000678,0.035580,0.135063
7,1676255,10,5.915660e-01,0.192000,0.046296,1.000000e+00,0.890288,0.420600,0.133098,0.778757
8,1676563,3,5.727571e-11,0.091667,0.000288,1.718271e-10,0.251135,0.000864,0.036808,0.106509
9,1676633,8,2.500009e-01,0.173750,0.087943,1.000000e+00,0.826800,0.689686,0.130735,0.773112



[OK] Guardado en: D:\Noveno Semestre\Tesis Informal\Codes\output\kapak_sie_process_indices_4methods.csv


In [12]:
print(f"Preguntas procesadas: {len(q_probs):,}")
print(f"Procesos unicos: {q_probs['contract_id'].nunique():,}")


Preguntas procesadas: 1,659
Procesos unicos: 169
